# Data Collection

## Template of df

In [ ]:
import pandas as pd

column_names = [
    'id', #
    'date',#
    'home_team',#
    'away_team',#
    'round',#
    'result', # h = home win, d = draw, a = away win #

    'home_halftime_score',
    'home_fulltime_score',
    'home_extratime_score',
    'home_penalty_score',

    'away_halftime_score',
    'away_fulltime_score',
    'away_extratime_score',
    'away_penalty_score',

    'home_points_before', # Points earned by home team before this match
    'away_points_before', # Points earned by away team before this match

    ###########  Missing to generate to DataFrame

    ###############TODO IMPORTANT: Consider if splitting tournaments into separate DataFrames
    ############### because the calculation of points is wrong


    'home_points_after', # Points earned by home team after this match
    'away_points_after', # Points earned by away team after this match

    'home_position_before', # table position before this match
    'away_position_before', # table position before this match
    'home_position_after', # table position after this match
    'away_position_after', # table position after this match

    'avg_home_goals_scored_last_5', # Avg goals scored by home team (last 5 games)
    'avg_home_goals_conceded_last_5', # Avg goals conceded by home team (last 5 games)
    'total_home_goals_scored_before', # Total goals scored by home team before this match
    'total_home_goals_conceded_before', # Total goals conceded by home team before this match
    
    'home_win_streak',
    'home_draw_streak',
    'home_loss_streak',

    'avg_away_goals_scored_last_5', # Avg goals scored by away team (last 5 games)
    'avg_away_goals_conceded_last_5', # Avg goals conceded by away team (last 5 games)
    'total_away_goals_scored_before', # Total goals scored by away team before this match
    'total_away_goals_conceded_before', # Total goals conceded by away team
    
    'away_win_streak',
    'away_draw_streak',
    'away_loss_streak',

    'last_meeting_result' # Result of previous head-to-head (0/1/2)
    ]

df = pd.DataFrame(columns=column_names)

print(df)
print(df.info())

Empty DataFrame
Columns: [id, date, home_team, away_team, round, result, home_halftime_score, home_halftime_score, home_goals_scored_last_5, home_goals_conceded_last_5, home_points, home_win_streak, home_draw_streak, home_loss_streak, away_goals_scored_last_5, away_goals_conceded_last_5, away_points, away_win_streak, away_draw_streak, away_loss_streak, last_meeting_result]
Index: []

[0 rows x 21 columns]
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 0 entries
Data columns (total 21 columns):
 #   Column                      Non-Null Count  Dtype 
---  ------                      --------------  ----- 
 0   id                          0 non-null      object
 1   date                        0 non-null      object
 2   home_team                   0 non-null      object
 3   away_team                   0 non-null      object
 4   round                       0 non-null      object
 5   result                      0 non-null      object
 6   home_halftime_score         0 non-null      o

## Dataframe Creation



### Function to process fixtures and return dataframe

In [23]:
import json
import pandas as pd
from IPython.display import display, HTML, Image
pd.set_option('display.max_columns', None)


def process_fixtures(filepath):
  """
  Reads fixture data from a JSON file, processes it, and returns a pandas DataFrame.

  Args:
      filepath (str): The path to the JSON file containing the fixture data.

  Returns:
      pd.DataFrame: A DataFrame containing the processed fixture data.
  """
  with open(filepath, 'r') as f:
      data = json.load(f)

  fixtures = data['response']

  rows = []
  for item in fixtures:
    goals_home = item['goals']['home']
    goals_away = item['goals']['away']
    result = 'h' if goals_home > goals_away else 'a' if goals_home < goals_away else 'd'
    row = {
          'id': item['fixture']['id'],
          'date': item['fixture']['date'],
          'Home': item['teams']['home']['name'],
          'Away': item['teams']['away']['name'],
          'round': item['league']['round'],
          'result': result,
          'home_halftime_score': item['score']['halftime']['home'],
          'home_fulltime_score': item['score']['fulltime']['home'],
          'home_extratime_score': item['score']['extratime']['home'],
          'home_penalty_score': item['score']['penalty']['home'],
          'away_halftime_score': item['score']['halftime']['away'],
          'away_fulltime_score': item['score']['fulltime']['away'],
          'away_extratime_score': item['score']['extratime']['away'],
          'away_penalty_score': item['score']['penalty']['away']
      }
    rows.append(row)

  df = pd.DataFrame(rows)
  return df

df21 = process_fixtures('./Fixtures/fixtures_2021.json')
df22 = process_fixtures('./Fixtures/fixtures_2022.json')

#turn date column into datetime
df21['date'] = pd.to_datetime(df21['date'])
df22['date'] = pd.to_datetime(df22['date'])

df21 = df21.sort_values(by='date').reset_index(drop=True)
df22 = df22.sort_values(by='date').reset_index(drop=True)

df = pd.concat([df21, df22], ignore_index=True)

print(df.head())

       id                      date                Home               Away  \
0  721722 2021-07-23 02:00:00+00:00      Club Queretaro       Club America   
1  721723 2021-07-24 00:00:00+00:00              Necaxa      Santos Laguna   
2  721724 2021-07-24 02:00:00+00:00           FC Juarez             Toluca   
3  721725 2021-07-24 21:00:00+00:00             Pachuca               Leon   
4  721726 2021-07-25 02:00:00+00:00  Guadalajara Chivas  Atletico San Luis   

          round result  home_halftime_score  home_fulltime_score  \
0  Apertura - 1      d                    0                    0   
1  Apertura - 1      a                    0                    0   
2  Apertura - 1      a                    1                    1   
3  Apertura - 1      h                    0                    4   
4  Apertura - 1      a                    0                    1   

   home_extratime_score  home_penalty_score  away_halftime_score  \
0                   NaN                 NaN           

### Feature Engineering

In [31]:
points = {team: 0 for team in pd.concat([df["Home"], df["Away"]]).unique()}
home_points_before = []
away_points_before = []

for _, row in df.iterrows():
    # Points before match
    home_points_before.append(points[row["Home"]])
    away_points_before.append(points[row["Away"]])
    
    # Determine match result
    if row["result"]=='h':  # Home win
        points[row["Home"]] += 3
    elif row["result"]=='a':  # Away win
        points[row["Away"]] += 3
    else:  # Draw
        points[row["Home"]] += 1
        points[row["Away"]] += 1

# Add to DataFrame
df["Home_Points_Before"] = home_points_before
df["Away_Points_Before"] = away_points_before

# reorder columns
df = df[['id', 'date', 'Home', 'Away', 'round', 'result',
         'Home_Points_Before',
         'home_halftime_score', 'home_fulltime_score', 'home_extratime_score', 'home_penalty_score',
         'Away_Points_Before',
         'away_halftime_score', 'away_fulltime_score', 'away_extratime_score', 'away_penalty_score',]]

# print number of rows and columns
print(f"DataFrame shape: {df.shape}")


DataFrame shape: (684, 16)


## Train/Test Split

# Model

## Gradient Boosting (XGBoost / LightGBM)

### Model Hyperparameters

In [ ]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=500,        # number of trees
    learning_rate=0.05,      # step size shrinkage
    max_depth=6,             # tree depth
    subsample=0.8,           # prevent overfitting
    colsample_bytree=0.8,    # column sampling
    eval_metric="mlogloss",  # for multi-class
    random_state=42
)

### Training

In [ ]:
model.fit(X_train, y_train)

### Evaluation

In [ ]:
y_pred_test = model.predict(X_test)
print(classification_report(y_test, y_pred_test))
print("Accuracy:", accuracy_score(y_test, y_pred_test))

### Prediction

In [ ]:
y_pred = model.predict(X_pred)
y_pred_proba = model.predict_proba(X_pred)  # gives win/draw/loss probabilities


## Neural Net (LSTM/GRU): For time-series trends

## Transformer or GNN: If using player networks or match sequences